# MIL mmBERT (120k): test-shift diagnosis + whole-document chunk inference + stacking / ensembling

**Steps**
1. **Diagnose the test shift.** The 120k run predicts 22.4% fusion on test, but the test domain mix with the train label rates implies ≈28.4% (on val the predicted mix matches the truth). Per-domain comparison and window-score / confidence distributions, val vs test.
2. **Whole-document inference.** 18% of texts are longer than 1024 tokens and head + tail drops their middle. Every text is split into overlapping 1024-token chunks, and the MIL window scores are aggregated over the whole document (most AI-like window of all chunks, most human-like window of all chunks, …) together with chunk-probability statistics.
3. **Stack / ensemble.** LightGBM on the validation rows (5-fold out-of-fold → honest score) with head + tail probabilities, chunk aggregates, and the old features model's probabilities. The best candidate by val macro-F1 writes `submission.csv`.

## Setup & config

In [ ]:
import os, gc, sys, time, math, random, socket, subprocess, warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, FileLink

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoConfig, AutoModel

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

QUICK = os.environ.get("QUICK", "0") == "1"

# must match the trained model (120k_mil_transformer.ipynb)
MODEL_NAME   = "jhu-clsp/mmBERT-base"
MAX_LEN      = 1024
WIN          = 32
WIN_STRIDE   = 16
MIL_CH       = 4
MIL_HIDDEN   = 256
MIL_TOPK     = 3
HEAD_DROPOUT = 0.1

RANDOM_STATE  = 42
VAL_N         = 1_000 if QUICK else 20_000
CHUNK_STRIDE  = 768         # chunks of MAX_LEN tokens, 256-token overlap
EVAL_BATCH    = 32
N_FOLDS_STACK = 5

TEXT, LANG, DOMAIN, LABEL = "text", "lang", "domain", "label"
CLASSES = ["human", "ai", "fusion"]
LANGS = ["en-EN", "de-DE"]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}
HUMAN, AI, FUSION = (LABEL2ID[c] for c in CLASSES)

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "| lightgbm", HAS_LGB, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])

@contextmanager
def timer(msg):
    t0 = time.time()
    print(f"[start] {msg} ...", flush=True)
    yield
    print(f"[done]  {msg}: {time.time() - t0:.1f}s", flush=True)

def macro_f1(y, pred):
    return f1_score(y, pred, average="macro")

def f1_row(y, pred):
    return {"macro_f1": macro_f1(y, pred),
            **{f"f1_{c}": v for c, v in zip(CLASSES, f1_score(y, pred, average=None, labels=range(3)))}}

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working"), Path("."), Path("..")]
def find_file(name, required=True):
    for root in SEARCH_ROOTS:
        if root.exists():
            hits = sorted(root.rglob(name))
            if hits:
                print(f"{name}: {hits[0]}")
                return hits[0]
    if required:
        raise FileNotFoundError(f"{name} not found under {SEARCH_ROOTS}: attach the run's output as an input")
    print(f"{name}: not found (optional)")
    return None

PT_PATH = find_file("mmbert_mil.pt")
MIL_NPZ = find_file("probs_mmbert_mil.npz")
OLD_NPZ = find_file("probs_mmbert_features.npz", required=False)

## Data (identical val split) & saved probabilities

In [ ]:
COMP = "ai-generated-text-detection-hw3-ucu-ai-2026"
def find_data_dir():
    for root in [Path("/kaggle/input"), Path("data"), Path("../data"), Path("../../data")]:
        if root.exists():
            hits = sorted(root.rglob("train.csv"))
            hits = [h for h in hits if COMP in str(h)] or hits
            if hits:
                return hits[0].parent
    raise FileNotFoundError("train.csv not found: add the competition data as an input")

DATA_DIR = find_data_dir()
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(next(iter(sorted(DATA_DIR.glob("sample_submission*.csv")))))
for df in (train, test):
    df[TEXT] = df[TEXT].fillna("").astype(str)

strat = train[LABEL] + "|" + train[LANG] + "|" + train[DOMAIN]
_, val_idx = train_test_split(train.index, test_size=VAL_N, stratify=strat, random_state=RANDOM_STATE)
val = train.loc[val_idx].reset_index(drop=True)
y_val = val[LABEL].map(LABEL2ID).values
checksum = pd.util.hash_pandas_object(val["id"], index=False).sum() % 10**8
print(f"val rows: {len(val):,} | test rows: {len(test):,} | val id checksum: {checksum} (expected 4108583)")

In [ ]:
def load_npz(path, keys):
    # returns {key: (val array, test array)} aligned to the val / test row order of this notebook
    z = np.load(path, allow_pickle=True)
    vpos = pd.Series(np.arange(len(z["val_ids"])), index=z["val_ids"])
    tpos = pd.Series(np.arange(len(z["test_ids"])), index=z["test_ids"])
    assert set(val["id"]) <= set(vpos.index) and set(test["id"]) <= set(tpos.index), f"{path.name}: ids do not match"
    vi, ti = vpos.loc[val["id"]].values, tpos.loc[test["id"]].values
    return {k: (z[f"val_{k}"][vi], z[f"test_{k}"][ti]) for k in keys}

mil = load_npz(MIL_NPZ, ["fused", "text", "mil", "w_max", "w_min"])
old = load_npz(OLD_NPZ, ["ht_fused", "ht_text", "ht_feat"]) if OLD_NPZ is not None else None

print("MIL head+tail fused, val macro-F1:", round(macro_f1(y_val, mil["fused"][0].argmax(1)), 4))
if old:
    print("old features model fused, val macro-F1:", round(macro_f1(y_val, old["ht_fused"][0].argmax(1)), 4))

## Step 1: where does test lose fusion?
`expected` = train label rate of that domain (labels are unknown on test). On val, predicted shares track the truth, so a large gap on test in one domain points at a shift there.

In [ ]:
sig = lambda x: 1 / (1 + np.exp(-np.asarray(x, dtype=np.float64)))
val_pred = mil["fused"][0].argmax(1)
test_pred = mil["fused"][1].argmax(1)
train_rate = pd.crosstab(train[DOMAIN], train[LABEL], normalize="index")[CLASSES]

def pred_share(df, pred):
    return pd.crosstab(df[DOMAIN].values, np.array(CLASSES)[pred], normalize="index").reindex(columns=CLASSES, fill_value=0)

vs, ts = pred_share(val, val_pred), pred_share(test, test_pred)
diag = pd.DataFrame({"n_test": test[DOMAIN].value_counts(), "test_share": test[DOMAIN].value_counts(normalize=True)})
for c in CLASSES:
    diag[f"{c}: train"] = train_rate[c]
    diag[f"{c}: val pred"] = vs[c]
    diag[f"{c}: test pred"] = ts[c]
diag["fusion gap (test pred - train)"] = diag["fusion: test pred"] - diag["fusion: train"]
diag["missing fusion rows"] = (-diag["fusion gap (test pred - train)"] * diag["n_test"]).round()
display(diag.sort_values("missing fusion rows", ascending=False).style.format(precision=3)
        .background_gradient(subset=["fusion gap (test pred - train)"], cmap="RdYlGn", vmin=-0.15, vmax=0.15))

exp = (diag["test_share"].values[:, None] * train_rate.loc[diag.index].values).sum(0)
print("overall test | expected:", dict(zip(CLASSES, exp.round(3))),
      "| predicted:", dict(zip(CLASSES, np.bincount(test_pred, minlength=3) / len(test_pred))))

### Is test fusion just *less confident*? Window scores and confidence, val vs test
If test has many texts in the middle of the window-score range or with low top-class probability (and val doesn't), the test fusion texts look different from training fusion (new generator / insertion style).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
for ax, key, title in zip(axes[:2], ["w_max", "w_min"], ["most AI-like window (max)", "most human-like window (min)"]):
    for name, arr, col in [("val", mil[key][0], "tab:blue"), ("test", mil[key][1], "tab:orange")]:
        ax.hist(sig(arr), bins=40, alpha=0.5, density=True, color=col, label=name)
    ax.set_yscale("log"); ax.set_title(f"sigmoid({title})"); ax.legend()
for name, P, col in [("val", mil["fused"][0], "tab:blue"), ("test", mil["fused"][1], "tab:orange")]:
    axes[2].hist(P.max(1), bins=40, alpha=0.5, density=True, color=col, label=name)
axes[2].set_yscale("log"); axes[2].set_title("top-class probability (fused)"); axes[2].legend()
plt.tight_layout(); plt.show()

def unsure(P, wmax, wmin):
    return pd.DataFrame({"top prob < 0.9": P.max(1) < 0.9,
                         "max window in (0.1, 0.9)": (sig(wmax) > 0.1) & (sig(wmax) < 0.9),
                         "min window in (0.1, 0.9)": (sig(wmin) > 0.1) & (sig(wmin) < 0.9)})
u = pd.concat({"val": unsure(mil["fused"][0], mil["w_max"][0], mil["w_min"][0]).groupby(val[DOMAIN].values).mean(),
               "test": unsure(mil["fused"][1], mil["w_max"][1], mil["w_min"][1]).groupby(test[DOMAIN].values).mean()}, axis=1)
display(u.style.format("{:.3f}").background_gradient(cmap="Oranges", axis=None))

# what do test texts the model is unsure about get predicted as?
m = mil["fused"][1].max(1) < 0.9
print(f"test rows with top prob < 0.9: {m.sum():,} ({m.mean():.1%}) | predicted as:",
      dict(zip(CLASSES, np.bincount(test_pred[m], minlength=3))))

## Step 2: whole-document chunk inference

In [ ]:
def internet_ok(host="huggingface.co", port=443, timeout=3):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

def find_local_model():
    root = Path("/kaggle/input")
    if root.exists():
        for cfg in sorted(root.rglob("config.json")):
            if "mmbert" in str(cfg).lower():
                return str(cfg.parent)
    return None

MODEL_SRC = find_local_model() or MODEL_NAME
if MODEL_SRC == MODEL_NAME and not internet_ok():
    raise RuntimeError("No internet and no mmBERT config / tokenizer in /kaggle/input.")
tokenizer = AutoTokenizer.from_pretrained(MODEL_SRC)
tokenizer.model_max_length = 10**9

_with = tokenizer("a")["input_ids"]
_without = tokenizer("a", add_special_tokens=False)["input_ids"]
_k = next(i for i in range(len(_with)) if _with[i:i + len(_without)] == _without)
PREFIX, SUFFIX = _with[:_k], _with[_k + len(_without):]
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
BODY_LEN = MAX_LEN - len(PREFIX) - len(SUFFIX)

def tokenize_full(texts, batch=2000):
    texts = list(texts)
    out = []
    for i in range(0, len(texts), batch):
        out += [np.asarray(x, dtype=np.int32) for x in tokenizer(texts[i:i + batch], add_special_tokens=False, verbose=False)["input_ids"]]
    return out

def chunks(ids):
    if len(ids) <= BODY_LEN:
        return [ids]
    starts = list(range(0, len(ids) - BODY_LEN + 1, CHUNK_STRIDE))
    if starts[-1] + BODY_LEN < len(ids):
        starts.append(len(ids) - BODY_LEN)
    return [ids[s:s + BODY_LEN] for s in starts]

def pad_batch(bodies):
    seqs = [np.concatenate([PREFIX, b, SUFFIX]).astype(np.int64) for b in bodies]
    L = int(math.ceil(max(len(s) for s in seqs) / 8) * 8)
    ids = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.from_numpy(s); mask[i, :len(s)] = 1
    return ids, mask

with timer("tokenize val / test (full length)"):
    tok_val, tok_test = tokenize_full(val[TEXT]), tokenize_full(test[TEXT])
print(f"longer than one chunk: val {np.mean([len(x) > BODY_LEN for x in tok_val]):.1%}, "
      f"test {np.mean([len(x) > BODY_LEN for x in tok_test]):.1%}")

In [ ]:
class MILFusionClassifier(nn.Module):
    # identical to 120k_mil_transformer.ipynb (needed to load the state dict)
    N_STATS = 6

    def __init__(self, backbone, n_classes=len(CLASSES)):
        super().__init__()
        self.backbone = backbone
        H = backbone.config.hidden_size
        n_mil = MIL_CH * self.N_STATS
        self.text_norm = nn.LayerNorm(2 * H)
        self.tok_scorer = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(H, MIL_HIDDEN), nn.GELU(),
                                        nn.Linear(MIL_HIDDEN, MIL_CH))
        self.mil_norm = nn.LayerNorm(n_mil)
        self.fuse = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(2 * H + n_mil, H), nn.GELU(),
                                  nn.Dropout(HEAD_DROPOUT), nn.Linear(H, n_classes))
        self.text_head = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(2 * H, n_classes))
        self.mil_head = nn.Sequential(nn.Linear(n_mil, 64), nn.GELU(), nn.Linear(64, n_classes))

    @staticmethod
    def window_scores(s, attention_mask):
        m = attention_mask.float().unsqueeze(1)
        if s.size(-1) < WIN:
            pad = WIN - s.size(-1)
            s, m = F.pad(s, (0, pad)), F.pad(m, (0, pad))
        num = F.avg_pool1d(s * m, WIN, WIN_STRIDE, ceil_mode=True)
        den = F.avg_pool1d(m, WIN, WIN_STRIDE, ceil_mode=True)
        w = num / den.clamp(min=1e-6)
        valid = den >= 0.25
        valid[..., 0] = valid[..., 0] | ~valid.any(-1)
        return w, valid

    @staticmethod
    def bag_stats(w, valid):
        v = valid.float().expand_as(w)
        n = v.sum(-1).clamp(min=1.0)
        w_hi = w.masked_fill(~valid, -1e4)
        w_max = w_hi.amax(-1)
        w_min = w.masked_fill(~valid, 1e4).amin(-1)
        top = w_hi.topk(min(MIL_TOPK, w.size(-1)), dim=-1).values
        tv = (top > -1e3).float()
        top_mean = (top * tv).sum(-1) / tv.sum(-1).clamp(min=1.0)
        mean = (w * v).sum(-1) / n
        std = torch.sqrt(((w - mean.unsqueeze(-1)) ** 2 * v).sum(-1) / n + 1e-6)
        share = (torch.sigmoid(w) * v).sum(-1) / n
        return torch.cat([w_max, top_mean, mean, w_min, std, share], dim=-1), w_max, w_min, w_hi

    def forward(self, input_ids, attention_mask):
        # returns the three heads' logits + the raw (un-normalised) MIL bag statistics
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        mean = (h * m).sum(1) / m.sum(1).clamp(min=1.0)
        text = self.text_norm(torch.cat([h[:, 0], mean], dim=-1).float())
        s = self.tok_scorer(h).float().transpose(1, 2)
        w, valid = self.window_scores(s, attention_mask)
        stats, _, _, _ = self.bag_stats(w, valid)
        z = self.mil_norm(stats)
        return self.fuse(torch.cat([text, z], dim=-1)), self.text_head(text), self.mil_head(z), stats

cfg = AutoConfig.from_pretrained(MODEL_SRC)
if hasattr(cfg, "reference_compile"):
    cfg.reference_compile = False
try:
    backbone = AutoModel.from_config(cfg, attn_implementation="sdpa")
except (TypeError, ValueError):
    backbone = AutoModel.from_config(cfg)
model = MILFusionClassifier(backbone)
state = torch.load(PT_PATH, map_location="cpu", weights_only=True)
state = {k.removeprefix("module."): v for k, v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
assert not missing, f"missing keys in {PT_PATH.name}: {missing[:10]}"
print("loaded", PT_PATH.name, "| unexpected keys:", unexpected[:5])
del state; gc.collect()
model = model.to(DEVICE).eval()
net = nn.DataParallel(model) if N_GPU > 1 else model

In [ ]:
STAT_NAMES = ["wmax", "topk", "wmean", "wmin", "wstd", "share"]
STAT_COLS = [f"{s}_c{k}" for s in STAT_NAMES for k in range(MIL_CH)]   # same order as bag_stats

@torch.no_grad()
def predict_chunks(tok_list):
    bodies, owner = [], []
    for d, ids in enumerate(tok_list):
        for c in chunks(ids):
            bodies.append(c); owner.append(d)
    order = np.argsort([len(b) for b in bodies])[::-1]
    probs = {k: np.zeros((len(bodies), 3), np.float32) for k in ["fused", "text", "mil"]}
    stats = np.zeros((len(bodies), len(STAT_COLS)), np.float32)
    for i in range(0, len(order), EVAL_BATCH):
        idx = order[i:i + EVAL_BATCH].copy()
        ids, mask = pad_batch([bodies[j] for j in idx])
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            out = net(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE))
        for k, lg in zip(probs, out[:3]):
            probs[k][idx] = F.softmax(lg.float(), -1).cpu().numpy()
        stats[idx] = out[3].float().cpu().numpy()
    df = pd.DataFrame(stats, columns=STAT_COLS)
    for k, P in probs.items():
        for j, c in enumerate(CLASSES):
            df[f"p_{k}_{c}"] = P[:, j]
    df["doc"] = owner
    print(f"{len(tok_list):,} docs -> {len(bodies):,} chunks")
    return df

def aggregate(ch):
    # chunk rows -> one row per document; "max of wmax" = most AI-like window anywhere in the text, etc.
    agg = {f"p_fused_{c}": ["mean", "max", "min", "std"] for c in CLASSES}
    agg.update({f"p_mil_{c}": ["mean", "max"] for c in CLASSES})
    agg.update({f"p_text_{c}": ["mean"] for c in CLASSES})
    for k in range(MIL_CH):
        agg.update({f"wmax_c{k}": ["max", "mean"], f"wmin_c{k}": ["min", "mean"], f"share_c{k}": ["mean", "max", "min"],
                    f"topk_c{k}": ["max"], f"wmean_c{k}": ["mean"], f"wstd_c{k}": ["max"]})
    g = ch.groupby("doc").agg(agg)
    g.columns = [f"ch_{a}_{b}" for a, b in g.columns]
    g["ch_n"] = ch.groupby("doc").size()
    g = g.fillna(0.0)
    g["ch_rule_fusion"] = sig(g["ch_wmax_c0_max"]) * (1 - sig(g["ch_wmin_c0_min"]))   # AI window AND human window present
    g["ch_rule_ai"] = sig(g["ch_wmin_c0_min"])
    g["ch_rule_human"] = 1 - sig(g["ch_wmax_c0_max"])
    return g.reset_index(drop=True)

with timer("chunk inference val"):
    agg_val = aggregate(predict_chunks(tok_val))
with timer("chunk inference test"):
    agg_test = aggregate(predict_chunks(tok_test))

### Chunk-level candidates (no learning)
On texts ≤ 1024 tokens nothing changes, so the table also shows the score on long texts only.

In [ ]:
def chunk_probs(g, how="mean"):
    return g[[f"ch_p_fused_{c}_{how}" for c in CLASSES]].values

long_val = np.array([len(x) > BODY_LEN for x in tok_val])
simple = {
    "MIL head+tail (baseline)": mil["fused"][0],
    "MIL chunks mean": chunk_probs(agg_val, "mean"),
    "MIL head+tail + chunks mean": (mil["fused"][0] + chunk_probs(agg_val, "mean")) / 2,
    "MIL rule (max / min window, whole doc)": agg_val[["ch_rule_human", "ch_rule_ai", "ch_rule_fusion"]].values,
}
if old:
    simple["MIL head+tail + old fused (avg)"] = (mil["fused"][0] + old["ht_fused"][0]) / 2
    simple["old fused (features run)"] = old["ht_fused"][0]
rows = [{"candidate": k, **f1_row(y_val, P.argmax(1)), "macro_f1_long_texts": macro_f1(y_val[long_val], P[long_val].argmax(1))}
        for k, P in simple.items()]
display(pd.DataFrame(rows).sort_values("macro_f1", ascending=False).style.format(precision=4)
        .background_gradient(subset=["macro_f1", "macro_f1_long_texts"], cmap="Greens"))

## Step 3: LightGBM stacker (5-fold OOF on val)
Two variants: MIL model only (head + tail + chunk aggregates) and + the old features model. Lang, domain and length are inputs too, so the stacker can trust the chunk view more on long texts.

In [ ]:
def prob_frame(P, prefix):
    return pd.DataFrame(P, columns=[f"{prefix}_{c}" for c in CLASSES])

def stack_matrix(split):
    i = 0 if split == "val" else 1
    df, tok, agg = (val, tok_val, agg_val) if split == "val" else (test, tok_test, agg_test)
    parts = [prob_frame(mil[k][i], f"ht_{k}") for k in ["fused", "text", "mil"]]
    parts.append(pd.DataFrame({"ht_wmax": mil["w_max"][i], "ht_wmin": mil["w_min"][i]}))
    parts.append(agg)
    X = pd.concat(parts, axis=1)
    X["log_tokens"] = np.log1p([len(x) for x in tok])
    X["lang"] = pd.Categorical(df[LANG].values, categories=LANGS)
    X["domain"] = pd.Categorical(df[DOMAIN].values, categories=sorted(set(train[DOMAIN]) | set(test[DOMAIN])))
    if old:
        for k in ["ht_fused", "ht_text", "ht_feat"]:
            X = pd.concat([X, prob_frame(old[k][i], f"old_{k}")], axis=1)
    return X

def make_gbm():
    if HAS_LGB:
        return lgb.LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=31, min_child_samples=40,
                                  subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=1.0,
                                  random_state=RANDOM_STATE, n_jobs=os.cpu_count(), verbose=-1)
    from sklearn.ensemble import HistGradientBoostingClassifier
    return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, categorical_features="from_dtype",
                                          random_state=RANDOM_STATE)

X_val, X_test = stack_matrix("val"), stack_matrix("test")
OLD_COLS = [c for c in X_val.columns if c.startswith("old_")]
skf = StratifiedKFold(n_splits=N_FOLDS_STACK, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(skf.split(np.zeros(len(val)), (val[LABEL] + "|" + val[DOMAIN]).values))

def oof(cols):
    P = np.zeros((len(val), 3), np.float32)
    for tr, te in FOLDS:
        P[te] = make_gbm().fit(X_val.iloc[tr][cols], y_val[tr]).predict_proba(X_val.iloc[te][cols])
    return P

STACKS = {"stacker: MIL head+tail + chunks": [c for c in X_val.columns if c not in OLD_COLS]}
if OLD_COLS:
    STACKS["stacker: MIL + chunks + old model"] = list(X_val.columns)
oof_p = {}
for name, cols in STACKS.items():
    with timer(name):
        oof_p[name] = oof(cols)

## Pick the best pipeline on val

In [ ]:
cands = {**simple, **oof_p}
cands.pop("old fused (features run)", None)
res = pd.DataFrame([{"pipeline": k, **f1_row(y_val, P.argmax(1)),
                     "macro_f1_long_texts": macro_f1(y_val[long_val], P[long_val].argmax(1))} for k, P in cands.items()])
res = res.sort_values("macro_f1", ascending=False).reset_index(drop=True)
display(res.style.format(precision=4).background_gradient(subset=["macro_f1", "f1_human", "f1_fusion", "macro_f1_long_texts"], cmap="Greens"))

BEST = res.loc[0, "pipeline"]
best_val_pred = cands[BEST].argmax(1)
base_pred = mil["fused"][0].argmax(1)
print(f"best: {BEST} | val macro-F1 {res.loc[0, 'macro_f1']:.4f} (baseline head+tail {macro_f1(y_val, base_pred):.4f})\n")
print(classification_report(y_val, best_val_pred, target_names=CLASSES, digits=4))

v = val.assign(base=base_pred, best=best_val_pred, y=y_val)
def rate(g, t, p, col):
    m = g["y"] == t
    return (g.loc[m, col] == p).mean() if m.any() else np.nan
dom = pd.DataFrame([{"lang": l, "domain": d, "n": len(g), "baseline": macro_f1(g["y"], g["base"]), "best": macro_f1(g["y"], g["best"]),
                     "h→f base": rate(g, HUMAN, FUSION, "base"), "h→f best": rate(g, HUMAN, FUSION, "best"),
                     "f→h base": rate(g, FUSION, HUMAN, "base"), "f→h best": rate(g, FUSION, HUMAN, "best")}
                    for (l, d), g in v.groupby([LANG, DOMAIN])]).assign(delta=lambda x: x["best"] - x["baseline"])
display(dom.sort_values("best").style.format(precision=4).background_gradient(subset=["delta"], cmap="RdYlGn", vmin=-0.05, vmax=0.05))

if BEST in STACKS and HAS_LGB:
    imp = pd.Series(make_gbm().fit(X_val[STACKS[BEST]], y_val).booster_.feature_importance("gain"), index=STACKS[BEST])
    grp = imp.groupby(lambda c: "old model" if c.startswith("old_") else "head+tail" if c.startswith("ht_")
                      else "chunk aggregates" if c.startswith("ch_") else "lang/domain/length").sum()
    display((grp / grp.sum()).sort_values(ascending=False).to_frame("gain share").style.format("{:.3f}"))
    display((imp / imp.sum()).sort_values(ascending=False).head(20).to_frame("gain share").style.format("{:.3f}"))

## Predict test & save submission
Stacker variants are refit on all validation rows. The step-1 shift check is repeated for the new predictions.

In [ ]:
def test_probs(name):
    if name in STACKS:
        return make_gbm().fit(X_val[STACKS[name]], y_val).predict_proba(X_test[STACKS[name]])
    return {
        "MIL head+tail (baseline)": lambda: mil["fused"][1],
        "MIL chunks mean": lambda: chunk_probs(agg_test, "mean"),
        "MIL head+tail + chunks mean": lambda: (mil["fused"][1] + chunk_probs(agg_test, "mean")) / 2,
        "MIL rule (max / min window, whole doc)": lambda: agg_test[["ch_rule_human", "ch_rule_ai", "ch_rule_fusion"]].values,
        "MIL head+tail + old fused (avg)": lambda: (mil["fused"][1] + old["ht_fused"][1]) / 2,
    }[name]()

test_P = test_probs(BEST)
test_pred_id = test_P.argmax(1)
sub = sample_sub[["id"]].merge(pd.DataFrame({"id": test["id"].values, "label": np.array(CLASSES)[test_pred_id]}), on="id", how="left")
assert sub["label"].notna().all() and len(sub) == len(sample_sub) and set(sub["label"]) <= set(CLASSES)
SUB_PATH = OUT_DIR / "submission.csv"
sub.to_csv(SUB_PATH, index=False)
print(f"pipeline: {BEST} | saved {SUB_PATH.resolve()} {sub.shape}")

np.savez_compressed(OUT_DIR / "probs_mil_chunks_stack.npz", val_ids=val["id"].values, test_ids=test["id"].values,
                    val_best=cands[BEST], test_best=test_P,
                    val_chunks_mean=chunk_probs(agg_val, "mean"), test_chunks_mean=chunk_probs(agg_test, "mean"))
agg_val.assign(id=val["id"].values).to_parquet(OUT_DIR / "chunk_agg_val.parquet")
agg_test.assign(id=test["id"].values).to_parquet(OUT_DIR / "chunk_agg_test.parquet")

ts_new = pred_share(test, test_pred_id)
display(pd.DataFrame({"fusion: train": train_rate["fusion"], "fusion: test pred (head+tail)": ts["fusion"],
                      "fusion: test pred (new)": ts_new["fusion"], "n_test": test[DOMAIN].value_counts()})
        .style.format(precision=3))
display(pd.DataFrame({"expected (train rates x test domains)": exp,
                      "test pred head+tail": np.bincount(test_pred, minlength=3) / len(test),
                      "test pred new": np.bincount(test_pred_id, minlength=3) / len(test)}, index=CLASSES).round(3))
os.chdir(OUT_DIR)
display(FileLink(SUB_PATH.name))